# Практическая работа №1. Часть 5Б
## Большие языковые модели: устройство и возможности

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ExPl0i/AI-10-NLP-LLM/blob/main/%D0%9F%D0%A0-%D0%91%D0%BE%D0%BB%D1%8C%D1%88%D0%B8%D0%B5%20%D1%8F%D0%B7%D1%8B%D0%BA%D0%BE%D0%B2%D1%8B%D0%B5%20%D0%BC%D0%BE%D0%B4%D0%B5%D0%BB%D0%B8.ipynb)

**Цель:** разобраться, чем большая языковая модель отличается от своего маленького GPT из части 4Г, откуда берутся ее возможности и ограничения и почему ей нужно столько памяти. Для этого заглянем внутрь настоящей открытой модели Qwen2.5-0.5B (494 миллиона параметров): посмотрим, на какие токены она режет русский текст и какие вероятности дает следующему токену, напишем свои способы выбора токена и генерацию с кэшем ключей и значений, сравним базовую модель с моделью-помощником, рассчитаем и измерим память, найдем голову внимания, которая связывает местоимение со словом.

> **Правило части.** Функции выбора следующего токена, цикл генерации, расчет памяти и разбор карт внимания пишем сами. Библиотека `transformers` нужна только для загрузки модели и токенизатора и для сверки с ее реализацией.

**Модели.** Обе модели компании Alibaba, лицензия Apache 2.0, загружаются с Hugging Face:
- `Qwen/Qwen2.5-0.5B` – **базовая модель**: только предобучение, продолжает любой текст;
- `Qwen/Qwen2.5-0.5B-Instruct` – **модель-помощник**: та же модель после дообучения на инструкциях и согласования с предпочтениями людей.

Каждая модель занимает около 1 ГБ на диске. Если преподаватель заранее запустил `prepare_data.py`, модели лежат в папке `models` рядом с ноутбуком и интернет не нужен.

**Два режима** выбираются в первой ячейке.
- `'легкий'` – компьютер в аудитории: модель на процессоре в 32-битном формате (около 2 ГБ оперативной памяти), короткие тексты по 40 токенов. Задания Б1 и Б2 выполняются за несколько минут. Задания Б3–Б5 тоже работают, но нужно около 6 ГБ свободной оперативной памяти и больше времени; на слабом компьютере их лучше делать в Colab.
- `'полный'` – Colab с видеокартой T4: модель на видеокарте в 16-битном формате, тексты по 120 токенов, все задания и задания повышенной сложности.

**Как работать с ноутбуком.** В ячейках с заданиями замените `...` своим кодом, подсказка в комментарии `# <--`. Проверки на маленьких примерах и расчеты должны совпасть с тем, что стоит после слова «ожидается», точно. Тексты, которые пишет модель, зависят от версии библиотек и от устройства, поэтому для них ожидания описаны словами.

## Как устроена большая языковая модель

**1. От своего GPT к большой модели.** Большая языковая модель – это тот же декодер, что вы собрали в части 4Г: эмбеддинги токенов, стопка одинаковых блоков (внимание с причинной маской и полносвязная часть с остаточными связями и нормализацией), выходной слой с вероятностями следующего токена. Меняются числа:

| | свой GPT (легкий режим) | Qwen2.5-0.5B | Qwen2.5-7B |
| --- | --- | --- | --- |
| блоков $L$ | 4 | 24 | 28 |
| длина вектора $d$ | 128 | 896 | 3584 |
| головы внимания (запросы / ключи и значения) | 4 / 4 | 14 / 2 | 28 / 4 |
| ширина полносвязной части $d_{ff}$ | 512 | 4864 | 18 944 |
| словарь $V$ | 1000 | 151 936 | 152 064 |
| окно контекста, токенов | 64 | 32 768 | 131 072 |
| параметров | 921 344 | 494 032 768 | 7 615 616 512 |

Кроме размера, в современных моделях есть четыре улучшения, которые не меняют принцип работы:
- **нормализация RMSNorm** вместо нормализации по слою: вектор делят на корень из среднего квадрата его чисел, без вычитания среднего;
- **вращательное позиционное кодирование RoPE** вместо синусоидального (повышенная сложность Т2.Б1.3): запросы и ключи поворачивают на угол, зависящий от позиции, и скалярное произведение зависит только от расстояния между токенами;
- **полносвязная часть SwiGLU** из трех матриц вместо двух: $\mathrm{FFN}(x) = W_2\big(\mathrm{SiLU}(W_1 x) \odot W_3 x\big)$, где $\odot$ – поэлементное умножение;
- **групповое внимание** (англ. grouped-query attention): у Qwen2.5-0.5B 14 голов запросов, но только 2 головы ключей и значений, каждую пару ключей и значений делят 7 голов запросов. Это в 7 раз уменьшает кэш ключей и значений (п. 5).

Число параметров блока Qwen2.5 складывается из внимания ($d^2 + d$ на запросы со сдвигом, по $d \cdot d_{kv} + d_{kv}$ на ключи и на значения, где $d_{kv} = h_{kv} \cdot d_{head}$, и $d^2$ на выход), полносвязной части ($3 \cdot d \cdot d_{ff}$) и двух нормализаций ($2d$). Вместе с эмбеддингами $V \cdot d$ (у маленькой модели выходной слой делит с ними веса) и последней нормализацией $d$ получается ровно 494 032 768. Обратите внимание: эмбеддинги – 136 миллионов, это 28 % всей модели. Огромный словарь нужен, чтобы одна модель хорошо работала с десятками языков и с программным кодом.

**2. Законы масштабирования.** Каплан и соавторы (2020) обнаружили, что потери языковой модели на проверке убывают как степенная функция от числа параметров $N$, объема обучающего текста $D$ в токенах и затраченных вычислений $C$, причем на протяжении многих порядков величины. Вычисления на обучение оцениваются формулой $C \approx 6ND$ операций с плавающей точкой: 2 операции на параметр на токен в прямом проходе и 4 в обратном. Хоффман и соавторы (2022) нашли, как при заданных вычислениях лучше распределить их между размером модели и объемом текста: около **20 токенов обучающего текста на параметр**. Для своего GPT это $20 \cdot 921\,344 \approx 18$ миллионов токенов, а он видел за 1500 шагов только $1500 \cdot 32 \cdot 64 \approx 3$ миллиона. Qwen2.5 обучали на 18 триллионах токенов, для модели на 0,5 миллиарда параметров это около 36 000 токенов на параметр – в сотни раз больше правила. Так делают намеренно: дообучить маленькую модель дольше дороже на этапе обучения, но потом каждый ответ обходится дешевле.

**3. Три этапа обучения.**
1. **Предобучение.** Предсказание следующего токена на огромном корпусе: веб-страницы, книги, код, научные статьи (у Qwen2.5 – 18 триллионов токенов). Получается **базовая модель**: она знает язык и факты, но умеет только продолжать текст. На вопрос «Как убрать запотевание стекла?» она может продолжить его списком похожих вопросов, как на форуме.
2. **Дообучение на инструкциях.** Модель учат на парах «запрос – образцовый ответ» (у Qwen2.5 – больше миллиона примеров), записанных в особом формате диалога со служебными токенами `<|im_start|>` и `<|im_end|>`. Задача обучения та же – следующий токен, но потери считают только на токенах ответа.
3. **Согласование с предпочтениями людей.** Для одного запроса берут два ответа модели, человек (или другая модель) отмечает лучший, и модель учат чаще давать ответы, похожие на лучшие: методом DPO на парах ответов и обучением с подкреплением (у Qwen2.5 – около 150 000 пар и затем обучение с подкреплением). Так модель становится вежливой, полезной и отказывается от вредных просьб.

После второго и третьего этапов получается **модель-помощник**. Знания у нее почти все из предобучения, а дообучение меняет манеру: отвечать на вопрос, а не продолжать его.

**4. Окно контекста.** Внимание сравнивает каждый токен с каждым, поэтому число операций и память под матрицу весов внимания растут как квадрат длины текста (Т2.Б1.3). Для 32 768 токенов это больше миллиарда пар в каждой голове каждого слоя. Поэтому окно ограничено, а модель на более длинном тексте, чем видела при обучении, работает плохо. Окно расширяют дообучением на длинных текстах с изменением частот RoPE и вычислением внимания блоками без хранения всей матрицы. Текст за пределами окна модель просто не видит.

**5. Генерация и кэш ключей и значений.** Ответ пишется по одному токену: модель предсказывает распределение следующего токена, из него выбирается токен, дописывается к тексту, и все повторяется. Поэтому чат-бот печатает ответ постепенно. Если на каждом шаге заново прогонять весь текст, работа растет как квадрат длины ответа. Но ключи и значения старых токенов не меняются (причинная маска: прошлое не смотрит на будущее), поэтому их сохраняют в **кэше ключей и значений** и на каждом шаге считают запрос, ключ и значение только нового токена. Размер кэша в байтах:
$$M_{kv} = 2 \cdot L \cdot h_{kv} \cdot d_{head} \cdot n_{tok} \cdot b,$$
где 2 – ключи и значения, $L$ – число блоков, $h_{kv}$ – число голов ключей и значений, $d_{head}$ – длина вектора головы, $n_{tok}$ – число токенов в окне, $b$ – байтов на число. Для Qwen2.5-0.5B в 16-битном формате это $2 \cdot 24 \cdot 2 \cdot 64 \cdot 2 = 12\,288$ байтов на токен, 0,4 ГБ на полное окно. Без группового внимания было бы в 7 раз больше.

**6. Выбор следующего токена.** Модель выдает оценки $z_1, \dots, z_V$ (логиты), из них функция softmax делает вероятности. Способы выбора:
- **жадный выбор** – токен с наибольшей оценкой. Ответ всегда одинаковый, но часто скучный и с повторами;
- **температура** $T$: вероятности $p_i = e^{z_i / T} / \sum_j e^{z_j / T}$. При $T < 1$ распределение острее (модель смелее выбирает главный вариант), при $T > 1$ – площе (чаще выпадают редкие токены). При $T \to 0$ остается только самый вероятный токен, поэтому температуру 0 понимают как жадный выбор;
- **отбор $k$ лучших** (top-k): оставить $k$ токенов с наибольшими оценками, остальным поставить оценку $-\infty$ (вероятность 0), вероятности оставшихся пересчитать;
- **отбор по накопленной вероятности** (top-p): упорядочить токены по убыванию вероятности и оставить самый короткий начальный набор, сумма вероятностей которого не меньше $p$. Если модель уверена, остается 1–2 токена, если сомневается – много.

В библиотеке `transformers` эти фильтры применяются в порядке «температура, top-k, top-p», и мы сделаем так же. Пример на шести токенах продолжения фразы «Водитель нажал на» с оценками 3,0; 2,2; 1,5; 1,0; 0,2; −1,0: при $T = 1$ вероятности 0,530; 0,238; 0,118; 0,072; 0,032; 0,010; top-k с $k = 3$ дает 0,598; 0,269; 0,133; top-p с $p = 0{,}9$ оставляет 4 токена (накопленные 0,530; 0,768; 0,886; 0,958).

**7. Память и квантование.** Память под веса модели равна числу параметров, умноженному на число байтов на параметр:

| формат | байтов на параметр | 0,5 млрд | 7 млрд | 70 млрд |
| --- | --- | --- | --- | --- |
| 32 бита (float32) | 4 | 2 ГБ | 28 ГБ | 280 ГБ |
| 16 бит (float16, bfloat16) | 2 | 1 ГБ | 14 ГБ | 140 ГБ |
| 8 бит | 1 | 0,5 ГБ | 7 ГБ | 70 ГБ |
| 4 бита | 0,5 | 0,25 ГБ | 3,5 ГБ | 35 ГБ |

Здесь 1 ГБ = $10^9$ байтов. **Квантование** – хранение весов меньшим числом битов: веса делят на группы (например, по 64 числа), для группы хранят масштаб в 16 битах, а сами числа округляют до 8 или 4 битов. При вычислениях числа восстанавливают умножением на масштаб. Качество при 8 битах почти не меняется, при 4 битах немного падает. Сверх весов нужна память на кэш ключей и значений и на промежуточные результаты, поэтому к весам добавляют запас – в наших расчетах 20 %. Видеокарта T4 в Colab – 16 ГБ (свободно около 15): модель на 7 миллиардов параметров в 16 битах в нее не помещается, а после квантования до 4 битов (около 4 ГБ с масштабами) – помещается.

**8. Возможности.**
- **Обучение по примерам в запросе** (англ. in-context learning): несколько пар «вопрос – ответ» в начале запроса задают формат и задачу, и модель продолжает в том же духе без изменения весов. Это работает даже у базовой модели.
- **Рассуждение по шагам**: если модель сначала пишет промежуточные шаги решения, а потом ответ, точность на задачах со счетом и логикой растет. Каждый написанный токен становится частью контекста, на который модель опирается дальше.
- **Вызов внешних инструментов**: модель пишет запрос к калькулятору, поиску или программе в условленном формате, программа выполняет его и дописывает результат в контекст.
- **Изображения и звук**: отдельный кодировщик превращает картинку или запись в последовательность векторов, которые подаются в модель наравне с эмбеддингами токенов.
- **Смесь экспертов** (англ. mixture of experts) одной строкой: полносвязная часть блока заменяется набором из многих «экспертов», и для каждого токена маленькая сеть-маршрутизатор выбирает несколько из них, поэтому параметров много, а вычислений на токен мало.

**9. Ограничения.**
- **Выдуманные факты.** Модель выбирает правдоподобное продолжение, а не проверенный факт. Если факт редкий, она уверенно пишет похожий на правду, но неверный ответ.
- **Дата среза знаний.** Модель знает только то, что было в обучающих текстах до определенной даты, и не знает о событиях после нее.
- **Счет букв.** Модель видит не буквы, а токены (Т2.Б1.1): слово «переворот» для нее – два-три номера токенов, поэтому вопрос «сколько букв р в слове» для нее труднее, чем для человека.
- **Поиск по документам** как способ дать модели свежие и точные знания: по вопросу находят подходящие абзацы в базе документов (смысловой поиск из части 4Г), вставляют их в запрос, и модель отвечает с опорой на них. Это задание повышенной сложности.

**Нужное из PyTorch и transformers.**
- `AutoTokenizer.from_pretrained(путь)`, `AutoModelForCausalLM.from_pretrained(путь, dtype=torch.float16)` – загрузка токенизатора и модели в нужном формате (в версиях `transformers` до 4.56 аргумент называется `torch_dtype`, ноутбук учитывает это в своей функции `from_pretrained`); `model.to(DEVICE)`; `model.config` – параметры архитектуры;
- `tok(text).input_ids` – номера токенов; `tok.decode([i])` – текст одного токена; `tok(text, return_offsets_mapping=True)['offset_mapping']` – границы токенов в символах текста;
- `out = model(input_ids=ids, past_key_values=past, use_cache=True)`: `out.logits` – оценки размера (пакет, длина, $V$), `out.past_key_values` – кэш ключей и значений для следующего шага; `model(ids, output_attentions=True).attentions` – веса внимания всех слоев, у каждого размер (пакет, головы, длина, длина);
- `torch.topk(x, k)` – $k$ наибольших значений и их номера; `torch.sort(x, descending=True)`, `torch.cumsum(x, dim=-1)`; `x.masked_fill(mask, float('-inf'))`; `torch.multinomial(probs, 1, generator=gen)` – случайный выбор по распределению;
- `tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)` – запрос к модели-помощнику в формате диалога;
- `model.get_memory_footprint()` – память под веса модели в байтах; `torch.cuda.memory_allocated()` – занятая память видеокарты.

In [ ]:
# Ячейка подготовки: режим, проверка среды, загрузка базовой модели
MODE = 'легкий'          # 'легкий' – компьютер в аудитории; 'полный' – Colab с видеокартой

import gc
import math
import platform
import re
import subprocess
import sys
import time
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')   # графики сохраняются в .png и показываются через display
import matplotlib.pyplot as plt
from IPython.display import Image, display

assert MODE in ('легкий', 'полный'), "MODE должен быть 'легкий' или 'полный'"
IN_COLAB = 'google.colab' in sys.modules
DEVICE = 'cuda' if (MODE == 'полный' and torch.cuda.is_available()) else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32
NEW_TOKENS = 120 if MODE == 'полный' else 40
print('Python', platform.python_version(), '| PyTorch', torch.__version__, '| среда:', 'Google Colab' if IN_COLAB else 'свой компьютер', '| режим:', MODE)
if MODE == 'полный' and DEVICE == 'cpu':
    print('ВНИМАНИЕ: полный режим рассчитан на видеокарту, а ее нет; модель будет работать на процессоре в 32-битном формате')
if DEVICE == 'cuda':
    print('видеокарта:', torch.cuda.get_device_name(0), f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} ГБ')

def ensure(module, package, version):
    """Ставит библиотеку фиксированной версии, только если ее нет."""
    try:
        __import__(module)
    except ImportError:
        print(f'устанавливаю {package}=={version} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--retries', '1', '--timeout', '10', f'{package}=={version}'], check=False)

ensure('transformers', 'transformers', '4.51.3')
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
transformers.logging.set_verbosity_error()
print('transformers', transformers.__version__)

NAVY, BLUE, DEEP, GRAY, LIGHT = '#0F1B2D', '#4C9BFF', '#1F63C9', '#9AA0A8', '#EAF2FF'
BASE_NAME, CHAT_NAME = 'Qwen/Qwen2.5-0.5B', 'Qwen/Qwen2.5-0.5B-Instruct'

def model_path(name):
    """Папка models/<имя>, если модель сохранена заранее (prepare_data.py), иначе имя на Hugging Face."""
    local = Path('models') / name.split('/')[-1]
    return str(local) if (local / 'config.json').exists() else name

NEW_API = tuple(int(x) for x in re.findall(r'\d+', transformers.__version__)[:2]) >= (4, 56)

def from_pretrained(name, dtype, **kwargs):
    """Загрузка модели в формате dtype (в новых версиях transformers аргумент называется dtype, в старых – torch_dtype)."""
    kwargs['dtype' if NEW_API else 'torch_dtype'] = dtype
    return AutoModelForCausalLM.from_pretrained(model_path(name), **kwargs)

def load_model(name, dtype=DTYPE, device=DEVICE):
    """Модель в нужном формате; внимание «eager» – чтобы можно было получить веса внимания (Б5)."""
    return from_pretrained(name, dtype, attn_implementation='eager').to(device).eval()

def free(*names):
    """Удалить модели из памяти."""
    for n in names:
        globals().pop(n, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

t0 = time.time()
tok = AutoTokenizer.from_pretrained(model_path(BASE_NAME))
lm = load_model(BASE_NAME)
STOP = {tok.convert_tokens_to_ids(t) for t in ('<|endoftext|>', '<|im_end|>')}
print(f'базовая модель загружена за {time.time() - t0:.0f} с:', sum(p.numel() for p in lm.parameters()), 'параметров,', lm.dtype, 'на', DEVICE)
print('  ожидается: 494032768 параметров')

### Задание Б1. Токены и вероятности следующего токена

Допишите две функции:
- `show_tokens(tok, text)` – список текстов всех токенов фразы: номера токенов `tok(text).input_ids`, текст каждого – `tok.decode([номер])`;
- `next_token_table(model, tok, text, k=10)` – $k$ самых вероятных следующих токенов: прогнать номера токенов через модель, взять оценки последней позиции `out.logits[0, -1]`, превратить их в вероятности функцией softmax (в 32-битном формате, `.float()`), выбрать $k$ наибольших через `torch.topk` и вернуть список пар (текст токена, вероятность).

Затем посмотрите, на какие токены модель режет русские и английские фразы, и какие продолжения она предлагает для трех начал.

In [ ]:
def show_tokens(tok, text):
    return ...                                      # <-- текст каждого токена через tok.decode([номер])

@torch.no_grad()
def next_token_table(model, tok, text, k=10):
    ids = tok(text, return_tensors='pt').input_ids.to(model.device)
    logits = ...                                    # <-- оценки последней позиции, в float32
    probs = ...                                     # <-- вероятности всех V токенов
    top = ...                                       # <-- k наибольших: top.values и top.indices
    return [(tok.decode([i]), p) for p, i in zip(top.values.tolist(), top.indices.tolist())]


pairs = [('Водитель нажал на тормоз, и машина остановилась.', 'The driver pressed the brake and the car stopped.'),
         ('Искусственный интеллект', 'Artificial intelligence')]
for ru, en in pairs:
    t_ru, t_en = show_tokens(tok, ru), show_tokens(tok, en)
    print(f'{len(ru)} символов, {len(t_ru)} токенов:', ' | '.join(t_ru))
    print(f'{len(en)} символов, {len(t_en)} токенов:', ' | '.join(t_en))
print('  ожидается: русская фраза – больше токенов, чем английская того же смысла; частые слова – целиком или из 2–3 кусков')
print('переворот ->', show_tokens(tok, 'переворот'), '  ожидается: несколько кусков, а не 9 букв')
print('словарь:', len(tok), 'токенов', '  ожидается: 151665 (151643 обычных и 22 служебных)')

for text in ['Водитель нажал на', 'Столица России –', 'Дважды два равно']:
    table = next_token_table(lm, tok, text)
    print(f'\n«{text}»: сумма вероятностей десяти первых {sum(p for _, p in table):.3f}')
    for t, p in table:
        print(f'   {t!r:16} {p:.3f}')
print('\n  ожидается: вероятности по убыванию, сумма десяти первых меньше 1;',
      'среди первых продолжений – начало слов «тормоз» или «педаль», « Москва», « четыре» или « 4»')

**Вопрос Б1.** а) Почему русская фраза требует больше токенов, чем английская того же смысла? Как это связано с тем, на каких текстах строили словарь (Т2.Б1.1), и чем это плохо для русскоязычного пользователя (подумайте про окно контекста и цену ответа)? б) Почему модели трудно ответить, сколько букв «р» в слове «переворот»? в) Для какой из трех фраз модель увереннее всего и почему?

*Ваш ответ:*

### Задание Б2. Свои способы выбора токена и генерация с кэшем

Допишите функции по п. 6 теории; все они работают с одномерным тензором оценок длины $V$:
- `top_k_filter(logits, k)` – оценки ниже $k$-й по величине заменить на $-\infty$;
- `top_p_filter(logits, p)` – вероятности по убыванию, накопленные суммы; удалить токен, если накопленная сумма **до него** (без его собственной вероятности) уже не меньше $p$, то есть `cum - sorted_p >= p`; вернуть маску удаления на исходные места через `scatter` и заменить удаленные оценки на $-\infty$;
- `sample_next(logits, temperature, top_k, top_p, gen)` – при температуре 0 жадный выбор, иначе деление на температуру, фильтры, softmax и случайный выбор `torch.multinomial`;
- `generate_text(model, tok, prompt, n_tokens, ...)` – свой цикл генерации с кэшем ключей и значений: на первом шаге в модель подается весь запрос, на следующих – только последний выбранный токен вместе с кэшем `past_key_values` из прошлого шага.

Проверки: числа примера из теории, сверка фильтров с `TopKLogitsWarper` и `TopPLogitsWarper` из `transformers`, частоты при многократном выборе, совпадение своей жадной генерации с `model.generate`.

In [ ]:
def top_k_filter(logits, k):
    kth = ...                                       # <-- k-я по величине оценка
    return logits.masked_fill(..., float('-inf'))   # <-- что удаляем

def top_p_filter(logits, p):
    probs = torch.softmax(logits, dim=-1)
    sorted_p, order = torch.sort(probs, descending=True)
    cum = ...                                       # <-- накопленные суммы
    remove_sorted = ...                             # <-- сумма до токена уже не меньше p
    remove = torch.zeros_like(remove_sorted).scatter(0, order, remove_sorted)
    return logits.masked_fill(remove, float('-inf'))

def sample_next(logits, temperature=1.0, top_k=None, top_p=None, gen=None):
    logits = logits.float().cpu()
    if temperature == 0:
        return ...                                  # <-- жадный выбор
    logits = ...                                    # <-- температура
    if top_k:
        logits = top_k_filter(logits, top_k)
    if top_p is not None and top_p < 1:
        logits = top_p_filter(logits, top_p)
    probs = ...
    return int(...)                                 # <-- случайный выбор по probs

@torch.no_grad()
def generate_text(model, tok, prompt, n_tokens, seed=0, **sampling):
    gen = torch.Generator().manual_seed(seed)
    x = tok(prompt, return_tensors='pt').input_ids.to(model.device)
    past, new_ids = None, []
    for _ in range(n_tokens):
        out = model(input_ids=x, past_key_values=past, use_cache=True)
        past = ...                                  # <-- кэш для следующего шага
        nxt = sample_next(..., gen=gen, **sampling)  # <-- оценки последней позиции
        if nxt in STOP:
            break
        new_ids.append(nxt)
        x = ...                                     # <-- дальше подаем только новый токен
    return tok.decode(new_ids)


from transformers.generation.logits_process import TopKLogitsWarper, TopPLogitsWarper
z = torch.tensor([3.0, 2.2, 1.5, 1.0, 0.2, -1.0])
show = lambda x: [round(v, 3) for v in torch.softmax(x, dim=-1).tolist()]
print('T = 1:  ', show(z), '  ожидается [0.53, 0.238, 0.118, 0.072, 0.032, 0.01]')
print('T = 0.5:', show(z / 0.5), '  ожидается [0.785, 0.158, 0.039, 0.014, 0.003, 0.0]')
print('T = 2:  ', show(z / 2), '  ожидается [0.346, 0.232, 0.163, 0.127, 0.085, 0.047]')
print('top-k 3:', show(top_k_filter(z, 3)), '  ожидается [0.598, 0.269, 0.133, 0.0, 0.0, 0.0]')
print('top-p 0.9:', show(top_p_filter(z, 0.9)), '  ожидается [0.553, 0.249, 0.123, 0.075, 0.0, 0.0]')

g = torch.Generator().manual_seed(1)
big = torch.tensor(np.random.default_rng(1).normal(0, 3, 1000), dtype=torch.float32)
same_k = torch.equal(top_k_filter(big, 50), TopKLogitsWarper(top_k=50)(None, big[None])[0])
same_p = torch.equal(top_p_filter(big, 0.9), TopPLogitsWarper(top_p=0.9)(None, big[None])[0])
print('совпадает с transformers: top-k', same_k, '| top-p', same_p, '  ожидается True True')
print('top-p 0.9 оставляет', int(torch.isfinite(top_p_filter(big, 0.9)).sum()), 'токен из 1000', '  ожидается 21')

counts = np.bincount([sample_next(z, gen=g) for _ in range(5000)], minlength=6) / 5000
print('частоты при T = 1:', counts.round(3).tolist(), '  ожидается близко к вероятностям при T = 1 (отличие меньше 0.02)')
print('жадный выбор:', sample_next(z, temperature=0), '  ожидается 0')

prompt = 'Водитель нажал на тормоз, и'
mine = generate_text(lm, tok, prompt, 20, temperature=0)
ids = tok(prompt, return_tensors='pt').input_ids.to(lm.device)
ref = lm.generate(ids, attention_mask=torch.ones_like(ids), max_new_tokens=20, do_sample=False, repetition_penalty=1.0, pad_token_id=tok.eos_token_id)
ref = tok.decode(ref[0, ids.shape[1]:], skip_special_tokens=True)
print('своя жадная генерация:', repr(mine))
print('совпадает с model.generate:', mine == ref, '  ожидается True')

In [ ]:
# Готовая ячейка: один запрос при разных настройках выбора токена
prompt = 'Вечером, когда солнце село за лес,'
settings = [dict(temperature=0), dict(temperature=0.7), dict(temperature=1.0), dict(temperature=1.5),
            dict(temperature=1.0, top_k=10), dict(temperature=1.0, top_p=0.9)]
for s in settings:
    for seed in (0, 1):
        text = generate_text(lm, tok, prompt, NEW_TOKENS, seed=seed, **s)
        print(f'{s} seed={seed}:', prompt + text.replace(chr(10), ' '), '\n')
print('ожидается: при температуре 0 оба текста одинаковые и гладкие, но могут повторяться;',
      'с ростом температуры тексты разнообразнее, при 1.5 появляются бессвязные куски; top-k и top-p убирают самые нелепые токены')

In [ ]:
# Готовая ячейка: зачем нужен кэш ключей и значений
@torch.no_grad()
def generate_no_cache(model, tok, prompt, n_tokens):
    """Жадная генерация без кэша: каждый шаг заново прогоняет весь текст."""
    x = tok(prompt, return_tensors='pt').input_ids.to(model.device)
    for _ in range(n_tokens):
        nxt = int(model(input_ids=x, use_cache=False).logits[0, -1].argmax())
        x = torch.cat([x, torch.tensor([[nxt]], device=model.device)], dim=1)
    return x

long_prompt = ' '.join(['Водитель включил фары и поехал дальше по ночной дороге.'] * 15)
n = 30
for name, fn in [('с кэшем', lambda: generate_text(lm, tok, long_prompt, n, temperature=0)),
                 ('без кэша', lambda: generate_no_cache(lm, tok, long_prompt, n))]:
    t0 = time.time(); fn()
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    print(f'{name}: {time.time() - t0:.1f} с на {n} токенов')
print('ожидается: с кэшем заметно быстрее, и разница растет с длиной запроса и ответа')

**Вопрос Б2.** а) Почему при температуре 0 ответ всегда одинаковый, а при температуре 1 – разный при разных `seed`? б) Что происходит с текстом при температуре 1,5 и почему (посмотрите на формулу)? в) Чем отбор по накопленной вероятности лучше отбора фиксированного числа токенов, когда модель уверена и когда сомневается? г) Почему кэш ключей и значений не меняет текст, а только ускоряет генерацию?

*Ваш ответ:*

### Задание Б3. Базовая модель и модель-помощник

Допишите `chat_prompt(tok, question, system=None)` – запрос в формате диалога: список сообщений `[{'role': 'system', 'content': ...}, {'role': 'user', 'content': question}]` (системное сообщение – только если оно задано) и `tok.apply_chat_template(..., tokenize=False, add_generation_prompt=True)`. Затем задайте один вопрос трем способам: базовой модели как есть, базовой модели с двумя примерами «вопрос – ответ» в начале и модели-помощнику в формате диалога. Везде жадный выбор, чтобы различия объяснялись моделью, а не случайностью.

In [ ]:
def chat_prompt(tok, question, system=None):
    messages = ...                                  # <-- список сообщений
    return ...


if MODE == 'легкий':
    print('легкий режим: модель-помощник загружается в оперативную память рядом с базовой, нужно около 4 ГБ свободной памяти')
chat_tok = AutoTokenizer.from_pretrained(model_path(CHAT_NAME))
chat = load_model(CHAT_NAME)

question = 'Как быстро убрать запотевание лобового стекла в машине?'
print('Запрос к помощнику целиком, со служебными токенами:\n' + chat_prompt(chat_tok, question))
print('  ожидается: системное сообщение по умолчанию, служебные токены <|im_start|> и <|im_end|>, в конце «<|im_start|>assistant»')

few_shot = ('Вопрос: Что делать, если загорелся индикатор резерва топлива?\n'
            'Ответ: Заправиться на ближайшей заправке: топлива осталось на несколько десятков километров.\n\n'
            'Вопрос: Как часто менять моторное масло?\n'
            'Ответ: Обычно раз в год или каждые 10–15 тысяч километров, точнее – в руководстве по эксплуатации.\n\n'
            f'Вопрос: {question}\nОтвет:')
answers = {'базовая модель, вопрос как есть': generate_text(lm, tok, question, NEW_TOKENS, temperature=0),
           'базовая модель, два примера в запросе': generate_text(lm, tok, few_shot, NEW_TOKENS, temperature=0).split('\n\n')[0],
           'модель-помощник, формат диалога': generate_text(chat, chat_tok, chat_prompt(chat_tok, question), NEW_TOKENS, temperature=0)}
for name, text in answers.items():
    print(f'\n=== {name}:\n{text.strip()}')
print('\nожидается: базовая модель продолжает текст (новые вопросы, варианты ответа как в тесте, обрывки статьи);',
      'с двумя примерами она отвечает в том же формате; помощник сразу дает связный ответ по пунктам')

**Вопрос Б3.** а) Объясните различия трех ответов через три этапа обучения. б) Почему два примера в запросе заставили базовую модель отвечать, хотя ее веса не менялись? в) Что видит модель-помощник, когда вы пишете ей в чате? Зачем ей служебные токены?

*Ваш ответ:*

### Задание Б4. Память модели: формула и замер

Допишите функции по пп. 1, 5 и 7 теории:
- `qwen_params(V, d, L, h, h_kv, d_ff)` – число параметров модели Qwen2.5 с общими весами эмбеддингов и выходного слоя: внимание со сдвигами у запросов, ключей и значений и без сдвига у выхода, полносвязная часть из трех матриц, две нормализации на блок, эмбеддинги и последняя нормализация;
- `weights_gb(n_params, bits)` – память под веса в гигабайтах ($10^9$ байтов);
- `kv_cache_bytes(n_layers, n_kv_heads, head_dim, n_tokens, bytes_per_value)` – размер кэша ключей и значений;
- `fits_in_gpu(n_params, bits, gpu_gb=16, reserve=1.2)` – помещается ли модель с запасом 20 % в видеокарту.

Затем сверьте формулы с замером: число параметров загруженной модели, память под веса в 32- и 16-битном формате (`get_memory_footprint`), размер настоящего кэша после запроса.

In [ ]:
def qwen_params(V, d, L, h, h_kv, d_ff):
    d_kv = h_kv * (d // h)                                  # длина ключа (и значения) всех голов ключей вместе
    attn = ...                                      # <-- запросы, ключи, значения со сдвигом, выход без сдвига
    ffn = ...                                       # <-- три матрицы SwiGLU
    block = attn + ffn + 2 * d
    return ...                                      # <-- эмбеддинги, блоки, последняя нормализация

def weights_gb(n_params, bits):
    return ...                                      # <-- байты на параметр = bits / 8

def kv_cache_bytes(n_layers, n_kv_heads, head_dim, n_tokens, bytes_per_value=2):
    return ...                                      # <-- ключи и значения

def fits_in_gpu(n_params, bits, gpu_gb=16, reserve=1.2):
    return ...


cfg = lm.config
head_dim = getattr(cfg, 'head_dim', None) or cfg.hidden_size // cfg.num_attention_heads
n_formula = qwen_params(cfg.vocab_size, cfg.hidden_size, cfg.num_hidden_layers, cfg.num_attention_heads, cfg.num_key_value_heads, cfg.intermediate_size)
n_real = sum(p.numel() for p in lm.parameters())
print('параметров по формуле', n_formula, '| у модели', n_real, '  ожидается 494032768 494032768')
print('Qwen2.5-7B по формуле:', qwen_params(152064, 3584, 28, 28, 4, 18944) + 152064 * 3584,
      '  ожидается 7615616512 (у 7B выходной слой не делит веса с эмбеддингами, поэтому + V·d)')
print('weights_gb:', weights_gb(7e9, 16), weights_gb(7e9, 4), weights_gb(70e9, 16), '  ожидается 14.0 3.5 140.0')
print('кэш Qwen2.5-0.5B, 1 токен, 16 бит:', kv_cache_bytes(24, 2, 64, 1), '  ожидается 12288')
print(f'кэш на полное окно 32768 токенов: 0.5B – {kv_cache_bytes(24, 2, 64, 32768) / 1e9:.2f} ГБ, 7B – {kv_cache_bytes(28, 4, 128, 32768) / 1e9:.2f} ГБ',
      '  ожидается 0.40 и 1.88')

print('\nпомещается ли в видеокарту 16 ГБ (запас 20 %):')
print(f'{"модель":>8} | ' + ' | '.join(f'{b:>2} бит' for b in (32, 16, 8, 4)))
for n in (0.5e9, 7e9, 70e9):
    print(f'{n / 1e9:>6g} B | ' + ' | '.join(f'{weights_gb(n, b):6.1f}{"+" if fits_in_gpu(n, b) else "-"}' for b in (32, 16, 8, 4)))
print('  ожидается: 0.5B помещается везде; 7B – только 8 и 4 бита (7.0+ и 3.5+); 70B – нигде (35.0- даже в 4 битах)')

In [ ]:
# Готовая ячейка: замер памяти под веса и размера настоящего кэша
if MODE == 'легкий':
    free('chat')   # освобождаем оперативную память: модель-помощник снова нужна только в заданиях повышенной сложности
rows = []
for dtype, bits in [(torch.float32, 32), (torch.float16, 16)]:
    m = from_pretrained(BASE_NAME, dtype)
    rows.append((bits, weights_gb(n_real, bits), m.get_memory_footprint() / 1e9))
    del m; gc.collect()
print('формат | формула, ГБ | замер, ГБ')
for bits, f, r in rows:
    print(f'{bits:>3} бит | {f:11.3f} | {r:9.3f}')
print('  ожидается: 1.976 и 0.988 по формуле, замер совпадает до третьего знака')
if DEVICE == 'cuda':
    print(f'видеокарта занята на {torch.cuda.memory_allocated() / 1e9:.2f} ГБ', '  ожидается: около 2 ГБ (базовая модель и помощник по 1 ГБ)')

def cache_tensors(cache):
    """Тензоры кэша в разных версиях transformers."""
    if hasattr(cache, 'layers'):
        return [t for layer in cache.layers for t in (layer.keys, layer.values)]
    if hasattr(cache, 'key_cache'):
        return list(cache.key_cache) + list(cache.value_cache)
    return [t for kv in cache for t in kv]

with torch.no_grad():
    ids = tok(long_prompt, return_tensors='pt').input_ids.to(lm.device)
    cache = lm(input_ids=ids, use_cache=True).past_key_values
size = sum(t.numel() * t.element_size() for t in cache_tensors(cache))
b = torch.finfo(lm.dtype).bits // 8
formula = kv_cache_bytes(cfg.num_hidden_layers, cfg.num_key_value_heads, head_dim, ids.shape[1], b)
print(f'запрос {ids.shape[1]} токенов: кэш {size} байтов, по формуле {formula}', '  ожидается: числа совпадают')

**Вопрос Б4.** а) Поместится ли модель на 7 миллиардов параметров в видеокарту T4 на 16 ГБ в 16-битном формате? А после квантования до 4 битов? Покажите расчет. б) Почему у маленькой модели эмбеддинги занимают так много параметров, а у большой – мало? в) Почему при очень длинном диалоге памяти может не хватить, даже если веса помещаются?

*Ваш ответ:*

### Задание Б5. На кого смотрит местоимение

Возьмем фразу «Вчера мальчик взял книгу, потому что она была интересной.» Местоимение «она» относится к слову «книгу»: «мальчик» мужского рода. Найдем голову внимания, которая это «понимает».

Слово модели часто разбито на несколько токенов, поэтому сначала соберем внимание по словам. Допишите:
- `token_owner(tok, text)` – номера токенов и для каждого токена номер слова, которому он принадлежит (или −1 для пробелов и знаков препинания): слова – это совпадения регулярного выражения `\w+`, токен принадлежит слову, если их отрезки в тексте пересекаются (`начало_токена < конец_слова` и `конец_токена > начало_слова`);
- `word_attention(att, owner, query_word, n_words)` – для матрицы весов внимания одной головы `att` размера (токены, токены): усреднить строки токенов слова-запроса, затем сложить веса по токенам каждого слова.

Почему фраза начинается со слова «Вчера»: первый токен текста почти во всех моделях собирает большую долю внимания (его называют стоком внимания), и если бы первым стоял «Мальчик», это исказило бы картину.

In [ ]:
def token_owner(tok, text):
    enc = tok(text, return_offsets_mapping=True)
    words = [(m.start(), m.end(), m.group()) for m in re.finditer(r'\w+', text)]
    owner = []
    for s, e in enc['offset_mapping']:
        k = next((j for j, (ws, we, _) in enumerate(words) if ...), -1)  # <-- отрезки пересекаются
        owner.append(k)
    return enc['input_ids'], [w for _, _, w in words], owner

def word_attention(att, owner, query_word, n_words):
    rows = [i for i, k in enumerate(owner) if k == query_word]
    row = ...                                       # <-- среднее по строкам токенов слова-запроса
    out = torch.zeros(n_words)
    for j, k in enumerate(owner):
        if k >= 0:
            ...                                     # <-- складываем веса по словам
    return out


text = 'Вчера мальчик взял книгу, потому что она была интересной.'
ids, words, owner = token_owner(tok, text)
print(words, '  ожидается 9 слов: Вчера ... интересной')
print('токены по словам:', [(tok.decode([i]), k) for i, k in zip(ids, owner)])
demo = torch.full((4, 4), 0.25)
print(word_attention(demo, [0, 0, 1, -1], 1, 2).tolist(), '  ожидается [0.5, 0.25]')

with torch.no_grad():
    attentions = lm(torch.tensor([ids], device=lm.device), output_attentions=True).attentions
A = torch.stack([a[0].float().cpu() for a in attentions])          # (слои, головы, токены, токены)
q, boy, book = words.index('она'), words.index('мальчик'), words.index('книгу')
score = torch.zeros(A.shape[0], A.shape[1])
for l in range(A.shape[0]):
    for h in range(A.shape[1]):
        w = word_attention(A[l, h], owner, q, len(words))
        score[l, h] = w[book] - w[boy]
best = torch.topk(score.flatten(), 5).indices
print('\nголовы, где «она» сильнее всего смотрит на «книгу», а не на «мальчик»:')
for i in best.tolist():
    l, h = divmod(i, A.shape[1])
    w = word_attention(A[l, h], owner, q, len(words))
    print(f'  слой {l:2d}, голова {h:2d}: на «книгу» {w[book]:.2f}, на «мальчик» {w[boy]:.2f}')
print('  ожидается: есть головы, где на «книгу» приходится заметно больше внимания, чем на «мальчик»')
print(f'доля внимания на первый токен, среднее по всем головам и токенам: {A[:, :, 1:, 0].mean():.2f}', '  ожидается: большая доля, сток внимания')

L_best, H_best = divmod(best[0].item(), A.shape[1])
M = torch.stack([word_attention(A[L_best, H_best], owner, k, len(words)) for k in range(len(words))])
fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(M, cmap='Blues')
ax.set_xticks(range(len(words)), words, rotation=45, ha='right'); ax.set_yticks(range(len(words)), words)
ax.set_xlabel('на какое слово смотрит'); ax.set_ylabel('какое слово смотрит')
ax.set_title(f'слой {L_best}, голова {H_best}')
fig.tight_layout(); fig.savefig('attention_head.png', dpi=110); plt.close(fig)
display(Image('attention_head.png'))

text2 = 'Вчера мальчик взял книгу, потому что он был любопытным.'
ids2, words2, owner2 = token_owner(tok, text2)
with torch.no_grad():
    att2 = lm(torch.tensor([ids2], device=lm.device), output_attentions=True).attentions[L_best][0, H_best].float().cpu()
w2 = word_attention(att2, owner2, words2.index('он'), len(words2))
print(f'та же голова, «он»: на «мальчик» {w2[words2.index("мальчик")]:.2f}, на «книгу» {w2[words2.index("книгу")]:.2f}',
      '  ожидается: часто «он» смотрит на «мальчик», но не во всех головах – проверьте')

**Вопрос Б5.** а) Опишите карту внимания найденной головы: куда смотрят слова, кроме местоимения? б) Найденная голова на второй фразе смотрит на правильное слово? Что это говорит о том, «понимает» ли голова местоимения? в) Почему нельзя судить о модели по одной голове одного слоя?

*Ваш ответ:*

### Задания повышенной сложности (к части 5Б)

1. **Поиск по документам своими руками.** В файле `data/manual.txt` – учебное руководство по эксплуатации условного автомобиля, 21 абзац. Превратите абзацы в векторы моделью rubert-tiny2 (функция `embed` из части 4Г: выход `[CLS]`, нормированный к длине 1), для вопроса водителя найдите три абзаца с наибольшим косинусом и вставьте их в запрос модели-помощнику: «Ответь на вопрос, используя только текст руководства. Руководство: … Вопрос: …». Сравните ответы с руководством и без него на трех вопросах: «Какое давление должно быть в задних шинах при полной загрузке?», «Что делать, если загорелся красный значок масленки?», «Каким маслом заправлен двигатель?». Где модель без руководства выдумывает?
2. **Квантование до 8 бит на видеокарте T4** (только полный режим). Установите `bitsandbytes`, загрузите модель-помощник с `BitsAndBytesConfig(load_in_8bit=True)` и сравните `get_memory_footprint()` с 16-битной моделью. Объясните, почему память уменьшилась не вдвое (подсказка: что библиотека не квантует). Сравните ответы двух моделей на одни и те же вопросы при жадном выборе.
3. **Рассуждение по шагам.** Задайте модели-помощнику пять задач со счетом («В машине 50 литров бензина, расход 7 литров на 100 километров. Хватит ли на 600 километров?») в двух вариантах: с просьбой сразу дать ответ и с просьбой сначала рассуждать по шагам. Сравните число верных ответов и объясните результат через генерацию по одному токену.

In [ ]:
# Место для заданий повышенной сложности: руководство к автомобилю для поиска по документам
import urllib.request
MANUAL = Path('data/manual.txt')
if not MANUAL.exists():
    MANUAL.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve('https://raw.githubusercontent.com/ExPl0i/AI-10-NLP-LLM/main/data/manual.txt', MANUAL)
paragraphs = [p.strip() for p in MANUAL.read_text(encoding='utf-8').split('\n\n') if p.strip()]
print(len(paragraphs), 'абзац', '  ожидается 21')